# Notebook 1: Building the Telemetry Layer

Part of *Quantifying AI Risk*. Hour 1 of 3.

## Terms used in this notebook

Before running any code, here are the terms you will see throughout this notebook.

- **Event:** one decision the AI made. For example, one loan application scored, or one document classified.
- **Signal:** a technical measurement captured per event. We capture six per decision.
- **Governance dimension:** a category of AI behavior a regulatory framework cares about. Fairness, transparency, robustness, and so on.
- **Control family:** a high-level grouping of related requirements in a framework. NIST groups them under MANAGE, MAP, MEASURE, GOVERN. ISO uses Annex A. EU AI Act organizes by Article.
- **Classify:** assign a label to a signal value based on rules. We label each signal as healthy, warning, or critical.
- **Blind spot:** a governance dimension not being measured at all. It produces no evidence, so it cannot be scored.
- **Drift:** the gradual shift over time in the data the model is seeing, away from the data it was trained on.
- **Detection lag:** the number of events between the moment drift begins and the moment a signal crosses its threshold.
- **Kernel:** the Python process that executes the notebook and holds variables in memory.
- **JSON Lines:** a file format where each line of the file is one complete JSON record. Streamable and inspectable.

## Step 1. Set up the environment

This step imports the standard library modules used throughout the notebook.

In [1]:
import random
import json
from datetime import datetime, timezone, timedelta
from collections import defaultdict

# Seed for reproducibility. Remove in production.
random.seed(42)

print("Ready.")

Ready.


## Step 2. Build a single governance telemetry event

Each event captures six signals per decision: what the system decided, model confidence, inference latency, distribution drift from training, demographic parity flag, and memory utilization.

Fairness is captured per event as a flag but reported as a rolling rate over a window. A single decision cannot be fair or unfair on its own.

An *event* is a single instance of the AI system producing an output. One input received, one decision made, one response returned. Every event should emit a structured record at the moment of decision.

Pretend we have a loan approval model running at Acme Corp. Every time the model decides on a loan application, that is one event. We will capture six things per event:

- what the system decided
- how confident it was
- how long it took
- how far today's inputs are from the training distribution
- whether this decision preserved demographic parity with the rolling distribution
- how much memory the process was using

One note on fairness before we write the code. A single decision cannot be fair or unfair on its own. What we capture per event is a flag. What we report at the dashboard level is a rolling rate over a window. You will see this pattern again when we get to classification.

In [2]:
def emit_event(applicant_id, model_id="loan-approval-v2"):
    """
    One governance telemetry event. In production this lives inside the model
    serving layer and emits to a stream (Kafka, Kinesis, Pub/Sub). Here we
    simulate realistic values for a healthy, in-distribution system.
    """
    return {
        "timestamp":     datetime.now(timezone.utc).isoformat(),
        "model_id":      model_id,
        "applicant_id":  applicant_id,

        # what the system decided
        "decision":      random.choice(["APPROVED", "REJECTED"]),

        # how sure it was. Beta(8,2) means most decisions land in 0.75-0.95.
        "confidence":    round(random.betavariate(8, 2), 3),

        # how long it took. Tight gaussian around 180ms.
        "latency_ms":    round(random.gauss(180, 35), 1),

        # how far today's inputs are from training. Beta(2,40) keeps healthy
        # systems well below 0.10.
        "drift_score":   round(random.betavariate(2, 40), 4),

        # did this decision preserve parity? True ~97% of the time when healthy.
        "fairness_flag": random.random() > 0.03,

        # infrastructure stress. Gaussian around 512MB.
        "memory_mb":     round(random.gauss(512, 30), 1),
    }


event = emit_event("APP-001")
print(json.dumps(event, indent=2))

{
  "timestamp": "2026-05-28T03:37:38.700147+00:00",
  "model_id": "loan-approval-v2",
  "applicant_id": "APP-001",
  "decision": "APPROVED",
  "confidence": 0.654,
  "latency_ms": 211.3,
  "drift_score": 0.0066,
  "fairness_flag": true,
  "memory_mb": 528.3
}


## Step 3. Generate a stream of events

This step calls the emit function many times to produce a stream of synthetic events that represent the model running in production.

One event is a point. A stream is a shape. Let's look at the shape.

In [3]:
random.seed(42)
events = [emit_event(f"APP-{i:03d}") for i in range(100)]

confidences  = [e["confidence"]  for e in events]
latencies    = [e["latency_ms"]  for e in events]
drifts       = [e["drift_score"] for e in events]
memories     = [e["memory_mb"]   for e in events]
decisions    = [e["decision"]    for e in events]
fairness_rate = sum(1 for e in events if e["fairness_flag"]) / len(events)


def summary(name, values, healthy_check):
    mean = sum(values) / len(values)
    breached = sum(1 for v in values if healthy_check(v))
    print(f"  {name:14s}  mean={mean:<8.3f}  min={min(values):<8.3f}  max={max(values):<8.3f}  breaches={breached}")


print("100 events, healthy baseline\n")
print("Decisions:")
approvals = sum(1 for d in decisions if d == "APPROVED")
print(f"  approved={approvals}  rejected={100-approvals}\n")

print("Per-event signals:")
summary("confidence",  confidences, lambda v: v < 0.60)
summary("latency_ms",  latencies,   lambda v: v > 300)
summary("drift_score", drifts,      lambda v: v > 0.10)
summary("memory_mb",   memories,    lambda v: v > 600)

print("\nRolling-rate signal:")
print(f"  fairness_rate  {fairness_rate:.1%}  (healthy >= 95%)")

100 events, healthy baseline

Decisions:
  approved=47  rejected=53

Per-event signals:
  confidence      mean=0.793     min=0.509     max=0.997     breaches=5
  latency_ms      mean=178.144   min=95.600    max=290.800   breaches=0
  drift_score     mean=0.044     min=0.003     max=0.146     breaches=6
  memory_mb       mean=512.408   min=442.400   max=573.600   breaches=0

Rolling-rate signal:
  fairness_rate  95.0%  (healthy >= 95%)


## Step 4. Map signals to governance dimensions

Each signal maps to a governance dimension, and each dimension anchors to a control family in NIST, ISO, and the EU AI Act. We map at the control-family level because sub-controls change between framework versions.

A *governance dimension* is a category of AI behavior that a regulatory framework cares about like fairness, transparency, robustness, accountability, and so on. A *signal* is the technical measurement we capture per event. The signal-to-dimension map is the connection between the two.

This map is what makes telemetry governance-grade rather than ordinary monitoring. Ordinary monitoring captures signals and stops there. Governance telemetry captures the signal *and* labels which regulatory dimension that signal speaks to.

We anchor each dimension to a *control family* in the major frameworks. A control family is a high-level grouping of related requirements. For example, NIST AI RMF organizes its requirements under MANAGE, MAP, MEASURE, and GOVERN. ISO 42001 uses Annex A. EU AI Act organizes by Article.

We anchor at the control-family level rather than at specific sub-control numbers, because sub-controls change between framework versions and depend on your certification scope. Your ISO 42001 lead auditor, or equivalent, performs the sub-control mapping for your organization. The engineer's job is to emit the signal and label the dimension correctly.

In [4]:
GOVERNANCE_MAP = {
    "confidence": {
        "dimension":   "Reliability & Robustness",
        "anchors":     ["NIST AI RMF MANAGE", "ISO 42001 Annex A"],
        "healthy":     0.70,
        "critical":    0.50,
        "direction":   "higher_is_better",
    },
    "latency_ms": {
        "dimension":   "Operational Health",
        "anchors":     ["NIST AI RMF MANAGE", "ISO 42001 Annex A"],
        "healthy":     300,
        "critical":    500,
        "direction":   "lower_is_better",
    },
    "drift_score": {
        "dimension":   "Data & Model Integrity",
        "anchors":     ["NIST AI RMF MAP", "ISO 42001 Annex A"],
        "healthy":     0.10,
        "critical":    0.20,
        "direction":   "lower_is_better",
    },
    "fairness_flag": {
        "dimension":   "Fairness & Societal Impact",
        "anchors":     ["EU AI Act Article 10", "NIST AI RMF MAP"],
        "healthy":     0.95,   # rolling rate, not per-event
        "critical":    0.85,
        "direction":   "higher_is_better",
    },
    "memory_mb": {
        "dimension":   "Resilience & Continuity",
        "anchors":     ["NIST AI RMF MANAGE", "ISO 42001 Annex A"],
        "healthy":     600,
        "critical":    800,
        "direction":   "lower_is_better",
    },
    "decision": {
        "dimension":   "Transparency & Accountability",
        "anchors":     ["EU AI Act Article 13", "EU AI Act Article 14"],
        "healthy":     None,   # always recorded, no threshold
        "critical":    None,
        "direction":   "record_only",
    },
}


for signal, info in GOVERNANCE_MAP.items():
    print(f"{signal}")
    print(f"  dimension  {info['dimension']}")
    print(f"  anchors    {', '.join(info['anchors'])}")
    if info['direction'] == 'record_only':
        print(f"  threshold  always recorded")
    elif info['direction'] == 'lower_is_better':
        print(f"  healthy    <= {info['healthy']}")
        print(f"  critical   >  {info['critical']}")
    else:
        print(f"  healthy    >= {info['healthy']}")
        print(f"  critical   <  {info['critical']}")
    print()

confidence
  dimension  Reliability & Robustness
  anchors    NIST AI RMF MANAGE, ISO 42001 Annex A
  healthy    >= 0.7
  critical   <  0.5

latency_ms
  dimension  Operational Health
  anchors    NIST AI RMF MANAGE, ISO 42001 Annex A
  healthy    <= 300
  critical   >  500

drift_score
  dimension  Data & Model Integrity
  anchors    NIST AI RMF MAP, ISO 42001 Annex A
  healthy    <= 0.1
  critical   >  0.2

fairness_flag
  dimension  Fairness & Societal Impact
  anchors    EU AI Act Article 10, NIST AI RMF MAP
  healthy    >= 0.95
  critical   <  0.85

memory_mb
  dimension  Resilience & Continuity
  anchors    NIST AI RMF MANAGE, ISO 42001 Annex A
  healthy    <= 600
  critical   >  800

decision
  dimension  Transparency & Accountability
  anchors    EU AI Act Article 13, EU AI Act Article 14
  threshold  always recorded



## Step 5. Classify the stream against thresholds

Each per-event signal is labeled as healthy, warning, or critical against the thresholds defined in the map. Fairness is labeled against its rolling rate over a window, not per event.

To *classify* an event means to assign it a label based on rules. In our case, each per-event signal is labeled as healthy, warning, or critical, depending on whether its value falls within an expected range. Those expected ranges are called thresholds, and the signal-to-dimension map defines them for every signal.

We read the stream of events against the map and label each signal accordingly. The fairness signal is the exception. Fairness is not classified per event because a single decision cannot be fair or unfair on its own. Fairness is classified against its rolling rate over a window of recent decisions, which is what fairness actually measures.

Classification is the bridge between the raw stream and the Bayesian scoring engine we will build in Hour 2. The scoring engine reads discrete evidence, not continuous values, so classification turns continuous signals into the input format the scoring engine expects.

In [5]:
def classify(signal_name, value, gov_map):
    info = gov_map[signal_name]
    if info["direction"] == "record_only":
        return "HEALTHY"

    healthy, critical = info["healthy"], info["critical"]

    if info["direction"] == "lower_is_better":
        if value <= healthy:  return "HEALTHY"
        if value <= critical: return "WARNING"
        return "CRITICAL"
    else:  # higher_is_better
        if value >= healthy:  return "HEALTHY"
        if value >= critical: return "WARNING"
        return "CRITICAL"


# Roll per-event signals up by dimension
per_event_signals = ["confidence", "latency_ms", "drift_score", "memory_mb"]
by_dimension = defaultdict(lambda: {"HEALTHY": 0, "WARNING": 0, "CRITICAL": 0})

for event in events:
    for signal in per_event_signals:
        status = classify(signal, event[signal], GOVERNANCE_MAP)
        by_dimension[GOVERNANCE_MAP[signal]["dimension"]][status] += 1

# Transparency is the audit trail itself. Always recorded, always healthy.
by_dimension["Transparency & Accountability"]["HEALTHY"] = len(events)

# Fairness runs against the rolling rate, reported separately.
fairness_status = classify("fairness_flag", fairness_rate, GOVERNANCE_MAP)


print("Dimension status, 100 events\n")

for dim in sorted(by_dimension.keys()):
    counts = by_dimension[dim]
    total = sum(counts.values())
    crit_pct = counts["CRITICAL"] / total * 100
    warn_pct = counts["WARNING"]  / total * 100

    if   crit_pct > 5:   overall = "CRITICAL"
    elif warn_pct > 15:  overall = "WARNING"
    else:                overall = "HEALTHY"

    print(f"  [{overall:8s}]  {dim}")
    print(f"              healthy={counts['HEALTHY']:3d}  warning={counts['WARNING']:3d}  critical={counts['CRITICAL']:3d}")

print(f"\n  [{fairness_status:8s}]  Fairness & Societal Impact")
print(f"              rolling parity rate = {fairness_rate:.1%}  (threshold: healthy >= 95%, critical < 85%)")

Dimension status, 100 events

  [HEALTHY ]  Data & Model Integrity
              healthy= 94  warning=  6  critical=  0
  [HEALTHY ]  Operational Health
              healthy=100  warning=  0  critical=  0
  [WARNING ]  Reliability & Robustness
              healthy= 78  warning= 22  critical=  0
  [HEALTHY ]  Resilience & Continuity
              healthy=100  warning=  0  critical=  0
  [HEALTHY ]  Transparency & Accountability
              healthy=100  warning=  0  critical=  0

  [HEALTHY ]  Fairness & Societal Impact
              rolling parity rate = 95.0%  (threshold: healthy >= 95%, critical < 85%)


## Step 6. Identify coverage gaps

This step compares the dimensions we have instrumented against a broader list of dimensions a mature governance framework expects. Dimensions not instrumented are blind spots and cannot be scored.

A *blind spot* in governance is a dimension of AI behavior that is not being measured at all. It is different from a signal that is performing badly. A bad signal produces evidence you can act on. A blind spot produces no evidence at all, which means it cannot be scored, defended, or improved.

The instrumentation we just built covers some governance dimensions and leaves others uninstrumented. The uninstrumented dimensions are our blind spots. Most governance assessments find this part uncomfortable, because the question shifts from "is our AI behaving well" to "do we even know whether our AI is behaving well in this area."

Below is an illustrative list of governance dimensions a mature framework might cover. Your organization's list may differ, depending on industry, jurisdiction, and the regulations you are exposed to. The exercise is the same in either case. Compare what you instrument against what your framework expects, and name the gaps explicitly.

In [6]:
# An illustrative dimension set. Your framework may define different boundaries.
DIMENSIONS = [
    "Reliability & Robustness",
    "Operational Health",
    "Data & Model Integrity",
    "Fairness & Societal Impact",
    "Resilience & Continuity",
    "Transparency & Accountability",
    "Human Oversight",
    "Content Safety & Truthfulness",
    "Security, Privacy & Data Governance",
    "Regulatory Alignment",
    "Explainability",
]

covered = {info["dimension"] for info in GOVERNANCE_MAP.values()}

covered_count = 0
for dim in DIMENSIONS:
    if dim in covered:
        print(f"  [covered]  {dim}")
        covered_count += 1
    else:
        print(f"  [  GAP  ]  {dim}")

pct = covered_count / len(DIMENSIONS) * 100
print(f"\nCoverage: {covered_count}/{len(DIMENSIONS)} dimensions ({pct:.0f}%)")
print(f"Gaps:     {len(DIMENSIONS) - covered_count} dimensions with no telemetry")
print("\nDimensions without telemetry cannot be scored. They are the blind spots.")

  [covered]  Reliability & Robustness
  [covered]  Operational Health
  [covered]  Data & Model Integrity
  [covered]  Fairness & Societal Impact
  [covered]  Resilience & Continuity
  [covered]  Transparency & Accountability
  [  GAP  ]  Human Oversight
  [  GAP  ]  Content Safety & Truthfulness
  [  GAP  ]  Security, Privacy & Data Governance
  [  GAP  ]  Regulatory Alignment
  [  GAP  ]  Explainability

Coverage: 6/11 dimensions (55%)
Gaps:     5 dimensions with no telemetry

Dimensions without telemetry cannot be scored. They are the blind spots.


## Step 7. Simulate drift and measure detection lag

This step injects a distribution shift partway through a longer simulated stream and measures how many events each signal takes to cross its healthy threshold.

*Drift* is the gradual shift in the input distribution your model is seeing over time, away from the distribution it was trained on. The model itself does not know this is happening. It continues to apply the patterns it learned, even when those patterns no longer match the world it is operating in. Good telemetry catches the shift while the model is still confidently wrong.

We will simulate a thousand events. For the first six hundred, the system runs healthy. At event 600, drift is injected. Imagine a new customer segment showing up, a policy change at the loan origination partner, or a market shift. The model does not register the change. The telemetry does.

Watch how the signals move together when drift arrives. This is the most important pattern to take away from this hour. Governance signals are correlated. When a system starts drifting, it typically also becomes less confident, slower, more resource-stressed, and less fair. A monitor that scores each signal independently produces four separate alerts and never recognizes that one underlying event caused all of them. A monitor that scores the signals jointly catches the pattern as a single event. Joint scoring is what we build in Hour 2.

*Detection lag* is the number of events between the moment drift actually begins and the moment a signal's rolling window crosses its healthy threshold. The smaller the lag, the sooner the system raises an alarm. The larger the lag, the more decisions the model makes under degraded conditions before anyone notices.

We measure detection lag for each signal individually. A single signal might take dozens of events to trip, because each signal only sees its own slice of the picture. When the same signals are combined probabilistically, the joint evidence crosses the threshold sooner than any one signal alone, because evidence from multiple correlated signals reinforces itself.

That gap between single-signal detection and joint-signal detection is the argument for Bayesian scoring, which is what Hour 2 builds.

In [7]:
def emit_event_with_scenario(event_num):
    """
    Events 0-599:    healthy, in-distribution
    Events 600-999:  drift has arrived, every signal degrades together
    """
    event = {
        "timestamp":    (datetime.now(timezone.utc) + timedelta(minutes=event_num)).isoformat(),
        "model_id":     "loan-approval-v2",
        "applicant_id": f"APP-{event_num:04d}",
        "decision":     random.choice(["APPROVED", "REJECTED"]),
    }

    if event_num < 600:
        # Healthy. Matches the baseline we saw earlier.
        event["confidence"]    = round(random.betavariate(8, 2), 3)
        event["latency_ms"]    = round(random.gauss(180, 35), 1)
        event["drift_score"]   = round(random.betavariate(2, 40), 4)
        event["fairness_flag"] = random.random() > 0.03
        event["memory_mb"]     = round(random.gauss(512, 30), 1)
    else:
        # Drift scenario. Every signal degrades.
        event["confidence"]    = round(random.betavariate(3, 3), 3)     # much less sure
        event["latency_ms"]    = round(random.gauss(320, 80), 1)        # slower, noisier
        event["drift_score"]   = round(random.betavariate(8, 10), 4)    # clearly elevated
        event["fairness_flag"] = random.random() > 0.15                 # parity degrading
        event["memory_mb"]     = round(random.gauss(620, 50), 1)        # more pressure
    return event


random.seed(42)
stream = [emit_event_with_scenario(i) for i in range(1000)]


def window_stats(events, label):
    n = len(events)
    print(f"  {label}  (n={n})")
    print(f"    confidence    {sum(e['confidence']  for e in events)/n:.3f}")
    print(f"    drift_score   {sum(e['drift_score'] for e in events)/n:.4f}")
    print(f"    fairness_rate {sum(1 for e in events if e['fairness_flag'])/n:.1%}")
    print(f"    latency_ms    {sum(e['latency_ms']  for e in events)/n:.0f}")
    print(f"    memory_mb     {sum(e['memory_mb']   for e in events)/n:.0f}")
    print()


print("Stream summary: before and after drift injection\n")
window_stats(stream[:600],  "events 0-599    (healthy)")
window_stats(stream[600:],  "events 600-999  (drift)")

print("Every signal moves in the same direction when drift arrives.")
print("Confidence drops. Drift score rises. Fairness degrades. Latency climbs.")
print("Memory pressure grows. These signals are correlated in production.")

Stream summary: before and after drift injection

  events 0-599    (healthy)  (n=600)
    confidence    0.799
    drift_score   0.0487
    fairness_rate 97.5%
    latency_ms    181
    memory_mb     511

  events 600-999  (drift)  (n=400)
    confidence    0.512
    drift_score   0.4435
    fairness_rate 88.2%
    latency_ms    322
    memory_mb     621

Every signal moves in the same direction when drift arrives.
Confidence drops. Drift score rises. Fairness degrades. Latency climbs.
Memory pressure grows. These signals are correlated in production.


In [8]:
DRIFT_START = 600
window = 50
lags = {}

for signal, info in GOVERNANCE_MAP.items():
    if info["direction"] == "record_only":
        continue

    for i in range(max(DRIFT_START, window), len(stream)):
        w = stream[i-window:i]
        if signal == "fairness_flag":
            value = sum(1 for e in w if e["fairness_flag"]) / window
        else:
            value = sum(e[signal] for e in w) / window

        if info["direction"] == "lower_is_better":
            breached = value > info["healthy"]
        else:
            breached = value < info["healthy"]

        if breached:
            lags[signal] = i - DRIFT_START
            break


print(f"Drift injected at event {DRIFT_START}. Rolling window = {window} events.\n")
print("Events after drift before each signal breaches its healthy threshold:\n")

for signal in sorted(lags, key=lags.get):
    print(f"  {signal:14s}  +{lags[signal]:3d} events")

print("\nDirect signals (drift, fairness) trip fastest. Downstream consequences")
print("(confidence, memory, latency) take longer. Hour 2 is about making the")
print("joint evidence cross threshold faster than any single signal can alone.")

Drift injected at event 600. Rolling window = 50 events.

Events after drift before each signal breaches its healthy threshold:

  drift_score     +  8 events
  fairness_flag   + 10 events
  confidence      + 23 events
  memory_mb       + 36 events
  latency_ms      + 42 events

Direct signals (drift, fairness) trip fastest. Downstream consequences
(confidence, memory, latency) take longer. Hour 2 is about making the
joint evidence cross threshold faster than any single signal can alone.


## Step 8. Persist the stream for Hour 2

This step writes the full stream of events to data/telemetry_stream.jsonl so Notebook 2 can read it. In production this layer is a durable event store such as Kafka or Kinesis. The principle is the same: write the event at the moment of decision, never reconstruct it after the fact.

The Jupyter kernel is the Python process that executes the code in this notebook and holds all the variables in memory. Everything we have built so far lives only in that memory. The moment the kernel shuts down, the stream is gone. That is fine for one notebook in isolation. It is not fine for a three-stage pipeline, because Hour 2 needs to read what Hour 1 wrote, even if you close the notebook and come back to it later.

We persist the stream as *JSON Lines*, a file format where each line of the file is one complete JSON object representing one event. The format is simple. Each line is a record. You can inspect it with standard command-line tools, and Notebook 2 can read it line by line without loading the entire file into memory at once.

In production this layer would be Kafka, Kinesis, or a durable event store. The technology changes. The principle stays the same. Write the event at the moment of decision, in a form that persists. Never reconstruct it after the fact.

In [9]:
from pathlib import Path

# Locate the data folder. If we are running from notebooks/ (Jupyter's default
# when launched from that folder), step up one level. Otherwise use ./data.
OUTPUT_DIR = Path("../data") if Path("../data").exists() else Path("data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "telemetry_stream.jsonl"

with open(OUTPUT_PATH, "w") as f:
    for event in stream:
        f.write(json.dumps(event) + "\n")

print(f"Wrote {len(stream)} events to {OUTPUT_PATH.resolve()}")
print(f"File size: {OUTPUT_PATH.stat().st_size:,} bytes")
print()
print("Notebook 2 will read from this same path.")

Wrote 1000 events to /Users/suneetamodekurty2o/Documents/projects/quantifying-ai-risk/data/telemetry_stream.jsonl
File size: 241,833 bytes

Notebook 2 will read from this same path.


## Your turn: close a coverage gap

The coverage analysis showed five governance dimensions with zero telemetry. Pick one of them and close the gap.

The five options:

- **Human Oversight.** Does the system escalate to a person when appropriate?
- **Content Safety and Truthfulness.** Does the system produce harmful or false output?
- **Security, Privacy, and Data Governance.** Is sensitive data being handled correctly?
- **Regulatory Alignment.** Can each decision be traced back to a specific regulatory requirement?
- **Explainability.** Can the system produce an explanation for its decision?

What to do:

1. **Design the signal.** Name it. Decide its type (per-event flag, continuous value, rolling rate). Define what healthy looks like and what critical looks like.
2. **Implement it.** Copy `emit_event()` into `emit_event_v2()` and add your new signal.
3. **Map it.** Add an entry to `GOVERNANCE_MAP_V2` with a defensible regulatory anchor.
4. **Classify it.** Run the pipeline on 100 fresh events.
5. **Justify the thresholds in writing.** Three to five sentences in the cell below the code. Name the specific risk each threshold guards against. Cite a regulatory anchor. Describe what would have to change in the world for you to revise the threshold.

This notebook does not provide a solution. The reason is operational. In production, every new governance signal requires engineering judgment about where to set the thresholds and how to defend them. A regulator will ask. A plaintiff's attorney will ask. The written justification is the artifact that survives both conversations. Practice writing it now, while the stakes are low.

In [10]:
# Your implementation

def emit_event_v2(applicant_id, model_id="loan-approval-v2"):
    # Copy the body of emit_event()
    # Add your new signal
    pass


GOVERNANCE_MAP_V2 = {
    # Copy GOVERNANCE_MAP entries
    # Add your new entry with dimension, anchors, thresholds, and direction
}


# Generate 100 fresh events, classify against GOVERNANCE_MAP_V2,
# and print coverage. Should be 7 of 11 dimensions instead of 6 of 11.


## Your threshold justification

*This is the cell referenced in step 5 of the exercise above. Write the written rationale for the thresholds you just chose in your code.*

*Every governance threshold has to be defensible, because someone will eventually ask why it is set where it is. A regulator during an audit. A plaintiff's attorney during discovery. An internal AI governance committee during a quarterly review. The written justification is the artifact that answers all three.*

*Three to five sentences. Three things to include.*

*1. The specific risk each threshold guards against.*
*2. At least one regulatory anchor (cite the framework, the article or clause, and what it requires).*
*3. What would have to change in the world before you would revise the thresholds.*

*This is the kind of written rationale that goes into an ISO 42001 technical documentation file. Treat the exercise accordingly.*

## What this notebook produced

A telemetry layer with the following components:

- An emitter that captures six signals on every decision
- A map that connects each signal to a governance dimension and a framework anchor
- A classifier that reads the stream against thresholds, handling per-event and rolling-rate signals differently
- A coverage view that names the blind spots
- A streaming simulation that shows what drift looks like and how fast each signal notices it
- A persisted stream that downstream notebooks will consume

This is the raw data layer of the pipeline. Everything in Hour 2 and Hour 3 reads from this stream. The Bayesian scoring engine in Hour 2 turns the stream into a continuously updated trust score with a credible interval. The Monte Carlo simulation in Hour 3 turns the trust score into a dollar-denominated risk distribution. The stream is the foundation. If the signal is not captured at the moment the decision is made, nothing downstream can recover it.

Recall the two cases the course opened with. Air Canada and Workday both lost on the same missing evidence. What this notebook built is the smallest working version of what they did not have. The methodology extends from here. Different industries will require different signals, different thresholds, different regulatory anchors. The architecture stays the same.

